In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
# %matplotlib qt
# %matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames


In [ ]:
%matplotlib widget


In [ ]:
output_dir = "/mnt/SpatialSequenceLearning/Simon/singleTrials"
data = {}
nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="INFO")


In [ ]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [ ]:
session_dirs = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)[0]
session_names = fullfnames2snames(session_dirs)

In [ ]:
framew_beh = analytics.get_analytics('BehaviorFramewise', session_names=session_names[:],)
framew_beh

In [ ]:
# s_id = '2024-12-11_17-42' #17
# s_id = '2024-12-11_17-42' #17
# trial_id = 34
time_col = 'frame_pc_timestamp'
# time_col = 'frame_ephys_timestamp'

# trial_data = framew_beh.xs(s_id, level='session_id').loc[framew_beh.xs(s_id, level='session_id')['trial_id'] == trial_id]
# trial_data = trial_data.set_index(time_col, drop=True, append=False)
# trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
# trial_data

In [ ]:
framew_beh.columns.tolist()

In [ ]:

def plot_one_trial(trial_data, s_id, trial_id):
    n_tmepoints = (trial_data.shape[0])
    plt.figure(figsize=(n_tmepoints*.02, 6))

    # main_metric = 'frame_velocity'
    # main_metric = 'frame_raw'
    main_metric = 'frame_RawYawPitch_abs_vel_sum'

    # which cue was shown
    color = 'orange' if trial_data['cue'].iloc[0] == 1 else 'purple'
    # if s_id < 8:
    #     color = 'white'

    # cue limits
    cue_entry_time = trial_data[(trial_data['track_zone'].isin(['visibleCue', 'nextToCue']))].index.values
    plt.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.1)

    # R1 limits
    R1_entry_time = trial_data[trial_data['track_zone']=='reward1Zone'].index.values
    plt.axvspan(R1_entry_time[0], R1_entry_time[-1], color='grey', alpha=0.2)

    
    # R2 limits
    R2_entry_time = trial_data[trial_data['track_zone']=='reward2Zone'].index.values
    plt.axvspan(R2_entry_time[0], R2_entry_time[-1], color='darkgrey', alpha=0.5)


    r1_thr = (trial_data['velocity_threshold_at_R1'] * trial_data['fps'])

    r1_frames = r1_thr[(r1_thr.index>R1_entry_time[0]) & (r1_thr.index<R1_entry_time[-1])].index
    # plt.plot(r1_thr.loc[r1_frames], linestyle='--', color='black', 
    plt.plot(r1_thr, linestyle='--', color='black', 
             alpha=.4, label='R1 thr.')
    below_thr_r1 = trial_data.loc[r1_frames, 'frame_RawYawPitch_abs_vel_sum'] <r1_thr.loc[r1_frames]
    # if below_thr_r1.any():
    #     plt.plot(trial_data.loc[r1_frames[below_thr_r1], 'frame_RawYawPitch_abs_velocity_sum'], color='green')
    
    
    r2_thr = (trial_data['velocity_threshold_at_R2'] * trial_data['fps'])
    r2_frames = r2_thr[(r2_thr.index>R2_entry_time[0]) & (r2_thr.index<R2_entry_time[-1])].index
    plt.plot(r2_thr.loc[r2_frames], linestyle='--', color='black', 
             alpha=.4, label='R2 thr.')
    below_thr_r2 = trial_data.loc[r2_frames, 'frame_RawYawPitch_abs_vel_sum'] <r2_thr.loc[r2_frames]
    # if below_thr_r2.any():
    #     plt.plot(trial_data.loc[r2_frames[below_thr_r2], 'frame_RawYawPitch_abs_velocity_sum'], color='green')
    
    print()

    # events
    labels = {
        'reward-sound_count': {
            'label': f'Sound',
            'color': 'green',
            'size': 80,
            'alpha': 0.7,
            'marker': '>'},
        'reward-valve-open_count': {
            'label': 'Reward',
            'color': 'green',
            'size': 80,
            'alpha': 0.7,
            'marker': 'o'},
        'lick_count': {
            'label': 'Lick',
            'color': 'black',
            'size': 50,
            'alpha': 0.4,
            'marker': '|'},
        'reward-removed_count': {
            'label': 'Reward-removed',
            'color': 'red',
            'size': 80,
            'alpha': 0.7,
            'marker': 'x'}
    }
    for event in labels.keys():
        event_data = trial_data[trial_data[event] >= 1]
        
        if len(event_data) == 0:
            continue
        
        # only check first sound event
        if event == 'reward-sound_count':
            # first check which reward zone this happened
            in_r1 = (event_data.index[0] >= R1_entry_time[0]) & (event_data.index[0] <= R1_entry_time[-1])
            in_r2 = (event_data.index[0] >= R2_entry_time[0]) & (event_data.index[0] <= R2_entry_time[-1])
            print(f"Sound event at {event_data.index.values}, in R1: {in_r1.any()}, in R2: {in_r2.any()}")
            
            if in_r1 and r1_frames[below_thr_r1].any():
                diff_to_thr = event_data.index[0] - r1_frames[below_thr_r1].values[0]
                # plot line between them (horizontal)
                plt.hlines(y=r1_thr.loc[event_data.index.values[0]],
                           xmin=r1_frames[below_thr_r1].values[0],
                           xmax=event_data.index[0],
                           color='black', )
                all_diffs.append(diff_to_thr)
            
            if in_r2 and r2_frames[below_thr_r2].any():
                diff_to_thr = event_data.index[0] - r2_frames[below_thr_r2].values[0]
                # plot line between them (horizontal)
                plt.hlines(y=r2_thr.loc[event_data.index.values[0]],
                           xmin=r2_frames[below_thr_r2].values[0],
                           xmax=event_data.index[0],
                           color='black', )
                print(f"  Diff to R2 thr: {diff_to_thr}")
                all_diffs.append(diff_to_thr)
                
            
        plt.scatter(
            event_data.index,
            event_data[main_metric],
            color=labels[event]['color'],
            s=labels[event]['size'],
            marker=labels[event]['marker'],
            label=f'{labels[event]["label"]}',
            alpha=labels[event]['alpha'],
            zorder=5  # Ensure markers appear on top
        )
        
    # what prop of the sum is forward driven, what prop is sideway or rotation driven
    frwd_prop = trial_data['frame_raw'].abs() / trial_data[main_metric]
    # sideway_prop = trial_data['frame_yaw'].abs() / trial_data[main_metric]
    # rotation_prop = trial_data['frame_pitch'].abs() / trial_data[main_metric]

    non_still_mask = trial_data[main_metric].abs() > 2.5
    sc = plt.scatter(trial_data.index[non_still_mask], 
                     np.ones_like(trial_data.index[non_still_mask]) * -25, 
                     c=frwd_prop[non_still_mask], s=4, cmap='afmhot', 
                     label='forward prop', alpha=0.9)
    cbar = plt.colorbar(sc)
    cbar.set_label("Proport. Forward vs Rotation")

    
    

    # main metrics
    plt.plot(trial_data[main_metric], zorder=1, color='darkblue', label='abs-ball-summed')
    plt.plot(trial_data['frame_RawYawPitch_abs_acc_sum_500msMedian'], zorder=1, color='darkblue', alpha=0.5, linestyle='-', label='ball-summed=smoothed')
    # ax_main.plot(trial_data['frame_pitch'].abs(), zorder=1, color='blue', alpha=0.3, linestyle=':', label='ball-sideways')
    # plt.plot(trial_data['frame_YawPitch_abs_vel_500msMedian'].abs(), zorder=1, color='purple', alpha=0.3, linestyle=':', label='ball-rotation-smoothed')
    # plt.plot(trial_data['frame_YawPitch_abs_acc_500msMedian'], zorder=1, color='purple', alpha=0.4, linestyle=':', label='ball-rotation-sm-acc')
    
    # plt.plot(trial_data['frame_raw_500msMedian'].abs(), zorder=1, color='blue', alpha=0.3, linestyle=':', label='ball-forward-smoothed')
    # plt.plot(trial_data['frame_raw_acc_500msMedian'], zorder=1, color='blue', alpha=0.4, linestyle=':', label='ball-forward-sm-acc')
    
    postfix = ''
    # plt.plot(trial_data['frame_raw'+postfix], zorder=1, color='lightblue', linestyle='-', label='forward')
    # plt.plot(trial_data['frame_raw_abs_acc_500msMedian'+postfix], zorder=1, color='lightblue', linestyle=':', label='forward')
    # plt.plot(trial_data['frame_yaw'+postfix], zorder=1, color='purple', linestyle='-', label='sideway', alpha=.6)
    # plt.plot(trial_data['frame_yaw_acc'+postfix], zorder=1, color='purple', linestyle=':', label='sideway_acc')
    # plt.plot(trial_data['frame_pitch'+postfix], zorder=1, color='darkgreen', linestyle='-', label='rotation', alpha=.6)
    # plt.plot(trial_data['frame_pitch_acc'+postfix], zorder=1, color='darkgreen', linestyle=':', label='rotation_acc')
    
    # postfix = '_500msMedian'
    # sm_ryp = trial_data[['frame_raw'+postfix, 'frame_yaw'+postfix, 'frame_pitch'+postfix]].abs().sum(axis=1)
    # plt.plot(trial_data['frame_forward_prop']*100, zorder=1, color='black', linestyle=':', label='forward_prop', alpha=.6)
    # plt.scatter((trial_data['frame_raw'].index), trial_data['frame_raw'+postfix]/sm_ryp *100, zorder=1, color='gray', linestyle=':', label='forward_prop', alpha=.2, s=20)
    
    # draw only points where below velocity threshold is true
    trial_data.loc[trial_data['frame_below_velocity_thr']==0] = np.nan
    r_thr_mean = (r1_thr+r2_thr)/2
    plt.plot(trial_data['frame_below_velocity_thr']*r_thr_mean -3, zorder=1, color='green', linestyle='-', linewidth=3, label='below_vel_thr')
    print(trial_data['frame_below_velocity_thr'].value_counts())
    
    plt.ylim(-40,150)
    # plt.xlim(0,5)

    plt.xlabel('Time (s)')
    plt.ylabel('Velocity [cm/s]')
    plt.title(f'Session {s_id} - Trial {trial_id}')
    plt.legend(ncol=2)

    # remove top spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(False)
    # line at y = 0
    plt.axhline(0, color='black', linestyle='--', alpha=0.3)

plt.close('all')

all_diffs = []
# s_id = 17
s_id = framew_beh.index.get_level_values('session_id').unique()[0]
print(f"Plotting session {s_id}...")
# s_id = '2024-12-06_16-49'  # 14 Example session ID

for trial_id in framew_beh.xs(s_id, level='session_id')['trial_id'].dropna().unique():
    if trial_id <40:
        continue
    trial_data = framew_beh.xs(s_id, level='session_id').loc[framew_beh.xs(s_id, level='session_id')['trial_id'] == trial_id]
    trial_data = trial_data.set_index(time_col, drop=True, append=False)
    trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
    plot_one_trial(trial_data, s_id, trial_id)

    if trial_id >50:
        break

plt.show()
# plt.close("all")
all_diffs